# Industrial Surface Defect Detection Using Deep Learning
## Convolutional Autoencoder Training Pipeline on Google Colab (with GPU Acceleration)

This notebook trains an unsupervised Convolutional Autoencoder on the **MVTec Anomaly Detection (MVTec AD)** dataset (Category: **Bottle**).

**Key Steps:**
1. Setup GPU environment and install dependencies
2. Download and extract MVTec AD Bottle dataset
3. Build Convolutional Autoencoder (Encoder-Decoder with 8×8×64 bottleneck)
4. Train on defect-free ('normal') images
5. Calibrate anomaly threshold ($\mu + 3\sigma$ control limit)
6. Evaluate on official test set (ROC-AUC, Precision, Recall, F1, Pixel IoU)
7. Download trained `autoencoder_bottle.keras` and config files for your local Streamlit app

### Step 1: Verify GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q opencv-python pillow matplotlib scikit-learn

### Step 2: Download & Extract MVTec AD Bottle Dataset
We download the official MVTec AD bottle category archive (approx. 140 MB).

In [ ]:
import os
import tarfile
import urllib.request

DATA_DIR = 'data'
CATEGORY = 'bottle'
CATEGORY_DIR = os.path.join(DATA_DIR, CATEGORY)
os.makedirs(DATA_DIR, exist_ok=True)

# Direct mirror / official source
url = 'https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420937370-1629951468/bottle.tar.xz'
tar_path = 'bottle.tar.xz'

if not os.path.exists(CATEGORY_DIR):
    print('Downloading MVTec AD Bottle dataset (~140MB)...')
    try:
        urllib.request.urlretrieve(url, tar_path)
        print('Extracting archive...')
        with tarfile.open(tar_path, 'r:xz') as tar:
            tar.extractall(path=CATEGORY_DIR)
        print('Extraction complete!')
    except Exception as e:
        print('Automated download error:', e)
        print('If download URL requires auth, download bottle.tar.xz manually from: https://www.mvtec.com/company/research/datasets/mvtec-ad')
else:
    print(f'Dataset already present at {CATEGORY_DIR}')

### Step 3: Dataset Loader & Preprocessing

In [ ]:
import glob
import numpy as np
from PIL import Image

TARGET_SIZE = (256, 256)

def load_image(path, size=TARGET_SIZE):
    with Image.open(path) as img:
        img = img.convert('RGB').resize(size, Image.Resampling.BILINEAR)
        return np.array(img, dtype=np.float32) / 255.0

train_files = sorted(glob.glob(f'{CATEGORY_DIR}/train/good/*.png'))
print(f'Found {len(train_files)} normal training images.')

train_images = np.stack([load_image(f) for f in train_files], axis=0)

# 85% Train, 15% Validation split
np.random.seed(42)
n_samples = len(train_images)
indices = np.random.permutation(n_samples)
n_val = int(n_samples * 0.15)
val_imgs = train_images[indices[:n_val]]
train_imgs = train_images[indices[n_val:]]

print(f'Train shape: {train_imgs.shape}, Val shape: {val_imgs.shape}')

### Step 4: Build Convolutional Autoencoder Architecture

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

def build_autoencoder(input_shape=(256, 256, 3)):
    inputs = layers.Input(shape=input_shape, name='input_image')

    # Encoder
    x = layers.Conv2D(32, 3, strides=2, padding='same')(inputs)      # -> 128x128x32
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(64, 3, strides=2, padding='same')(x)           # -> 64x64x64
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(128, 3, strides=2, padding='same')(x)          # -> 32x32x128
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(256, 3, strides=2, padding='same')(x)          # -> 16x16x256
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Latent Bottleneck: 8x8x64 = 4096 dimensions
    latent = layers.Conv2D(64, 3, strides=2, padding='same', name='latent_bottleneck')(x)
    latent = layers.BatchNormalization()(latent)
    latent = layers.ReLU()(latent)

    # Decoder
    d = layers.Conv2DTranspose(256, 3, strides=2, padding='same')(latent) # -> 16x16x256
    d = layers.BatchNormalization()(d)
    d = layers.ReLU()(d)

    d = layers.Conv2DTranspose(128, 3, strides=2, padding='same')(d)     # -> 32x32x128
    d = layers.BatchNormalization()(d)
    d = layers.ReLU()(d)

    d = layers.Conv2DTranspose(64, 3, strides=2, padding='same')(d)      # -> 64x64x64
    d = layers.BatchNormalization()(d)
    d = layers.ReLU()(d)

    d = layers.Conv2DTranspose(32, 3, strides=2, padding='same')(d)      # -> 128x128x32
    d = layers.BatchNormalization()(d)
    d = layers.ReLU()(d)

    outputs = layers.Conv2DTranspose(3, 3, strides=2, padding='same', activation='sigmoid')(d) # -> 256x256x3

    autoencoder = models.Model(inputs=inputs, outputs=outputs, name='DefectAutoencoder')
    autoencoder.compile(optimizer=optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return autoencoder

model = build_autoencoder()
model.summary()

### Step 5: Train the Autoencoder with Callbacks

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import json

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint('autoencoder_bottle.keras', monitor='val_loss', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

history = model.fit(
    x=train_imgs,
    y=train_imgs,
    validation_data=(val_imgs, val_imgs),
    epochs=50,
    batch_size=16,
    callbacks=callbacks
)

# Save training history
history_dict = {
    'loss': [float(x) for x in history.history['loss']],
    'val_loss': [float(x) for x in history.history['val_loss']]
}
with open('training_history.json', 'w') as f:
    json.dump(history_dict, f, indent=4)

### Step 6: Calibrate Decision Threshold ($\mu + 3\sigma$ Control Limit)

In [ ]:
val_preds = model.predict(val_imgs)
val_scores = []
for i in range(len(val_imgs)):
    diff = np.mean(np.abs(val_imgs[i] - val_preds[i]), axis=-1)
    k_pix = max(1, int(diff.size * 0.01))
    top_k = np.partition(diff.flatten(), -k_pix)[-k_pix:]
    val_scores.append(float(np.mean(top_k)))

mean_val = float(np.mean(val_scores))
std_val = float(np.std(val_scores))
recommended_threshold = float(mean_val + 3.0 * std_val)

config = {
    'category': 'bottle',
    'mean_normal_score': mean_val,
    'std_normal_score': std_val,
    'recommended_threshold': recommended_threshold,
    'pixel_threshold': float(recommended_threshold * 0.7)
}
with open('threshold_config.json', 'w') as f:
    json.dump(config, f, indent=4)

print(f'Calibrated Anomaly Threshold: {recommended_threshold:.5f}')

### Step 7: Plot Loss Curves and Visualizations

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history_dict['loss'], label='Training Loss (MSE)', lw=2)
plt.plot(history_dict['val_loss'], label='Validation Loss (MSE)', lw=2, linestyle='--')
plt.title('Autoencoder Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.savefig('loss_curve.png', dpi=200)
plt.show()

### Step 8: Download Trained Artifacts to Local Machine

In [ ]:
from google.colab import files

print('Downloading trained weights and configuration for Streamlit...')
files.download('autoencoder_bottle.keras')
files.download('threshold_config.json')
files.download('training_history.json')
files.download('loss_curve.png')
print('Place these files into the models/ and results/ directories of your local workspace!')